# Defining Pipeline Options

This notebook demonstrates the option-variable classes defined in
`pyBrainAnalyzIR.dataclasses.options_variables`.

These classes are intended to eventually replace the raw python values
(`int`, `float`, `bool`, `str`, ...) that are currently stored in the
`options` dictionary of the pipeline modules (see
`pyBrainAnalyzIR.pipelines.modules`).  Compared to a plain value, an option
variable additionally knows:

* its **current value** (validated every time it is changed),
* its **default value** (so it can always be reset),
* a **validation rule** (range, allowed set, enum membership, units, ...),
* a **help string** documenting what the option does,
* an optional **custom display** used when the option is printed.

Available classes:

| class | use for |
|---|---|
| `OptionVariable` | abstract base class - subclass this to make your own option type |
| `NumericOption` | ints / floats, optionally restricted to a range or to integers |
| `BooleanOption` | True / False flags |
| `StringOption` | free text, or text restricted to a set of allowed strings |
| `EnumOption` | a member of a custom `enum.Enum` type |
| `ChoiceOption` | any explicit list of allowed values (mixed types allowed) |
| `ListOption` | a list of values, each validated by another option |
| `QuantityOption` | physical quantities with units (e.g. `1 * units.Hz`) |

In [3]:
import enum
import pyBrainAnalyzIR
import pyBrainAnalyzIR.dataclasses.options_variables as options

import cedalion
units = cedalion.units

## The common interface

Every option type derives from the abstract `OptionVariable` class and
therefore shares the same interface, regardless of the kind of value it holds:

* `opt.value` - get / set the current value (validated on set)
* `opt.default` - the default value
* `opt.reset()` - restore the current value back to the default
* `opt.is_default` - `True` when the current value equals the default
* `opt.is_valid(x)` - test a candidate value without raising
* `opt.copy()` - deep copy of the option
* `opt.help` - the help text describing the option
* `opt.print_help()` - print name, value, description, help and default
* `print(opt)` - by default prints only the *current value*

## `NumericOption`

Use for integer or floating point options.  Restrictions:

* `minimum` / `maximum` - allowed range (use `inclusive=False` for a strict bound)
* `integer_only=True` - reject (or coerce) non-integer values
* `allow_none=True` - permit `None` as a valid "not set" value

This mirrors, e.g., the `butter_order` option of the `bandpass_filter` module,
which must be a positive integer.

In [4]:
# butter_order: an integer >= 1
butter_order = options.NumericOption(
    4,
    name='butter_order',
    minimum=1,
    integer_only=True,
    help='Order of the Butterworth filter. Higher values give a sharper '
         'roll-off but more ringing.',
)

print('current value :', butter_order)      # print shows only the value
print('default       :', butter_order.default)
print('is default    :', butter_order.is_default)

butter_order.value = 3                       # valid -> accepted
print('after change  :', butter_order, '| is_default:', butter_order.is_default)

butter_order.reset()                         # back to the default
print('after reset   :', butter_order)

current value : 4
default       : 4
is default    : True
after change  : 3 | is_default: False
after reset   : 4


In [5]:
# Invalid assignments raise a ValueError naming the offending option
for bad in (-1, 2.5, 'four'):
    try:
        butter_order.value = bad
    except (ValueError, TypeError) as err:
        print(f'{bad!r:>8} rejected -> {err}')

# ...or test without raising
print()
print('is_valid(10)  :', butter_order.is_valid(10))
print('is_valid(0)   :', butter_order.is_valid(0))

      -1 rejected -> butter_order: value must be >= 1 (got -1)
     2.5 rejected -> butter_order: value must be an integer (got 2.5)
  'four' rejected -> butter_order: value must be numeric (got 'four')

is_valid(10)  : True
is_valid(0)   : False


In [6]:
# A bounded float, e.g. the `ncomp` option of the pca_filter module which is
# the fraction of variance to remove (0-1).
ncomp = options.NumericOption(
    0.8,
    name='ncomp',
    minimum=0,
    maximum=1,
    help='Fraction of the variance removed by the PCA filter, or (if >1) the '
         'number of components to remove.',
)
print(ncomp)

try:
    ncomp.value = 1.5
except ValueError as err:
    print('rejected ->', err)

0.8
rejected -> ncomp: value must be <= 1 (got 1.5)


In [7]:
# `inclusive=False` makes the bound strict (e.g. must be strictly positive),
# and `allow_none=True` permits an explicit "not set" value.
tune = options.NumericOption(
    6.0, name='tune', minimum=0, inclusive=False,
    help='Tuning constant of the robust estimator; must be strictly positive.')
print('tune         :', tune)
print('is_valid(0)  :', tune.is_valid(0))    # 0 is excluded by inclusive=False

max_iter = options.NumericOption(
    None, name='max_iter', minimum=1, integer_only=True, allow_none=True,
    help='Maximum number of iterations. None means use the solver default.')
print('max_iter     :', max_iter)

tune         : 6.0
is_valid(0)  : False
max_iter     : None


## `BooleanOption`

Use for simple on/off flags, such as the `split_types` option of the
`pca_filter` module.  Only `True`/`False` (and the numeric literals `0`/`1`)
are accepted - a typo like `'yes'` is rejected rather than silently being
treated as truthy.

In [8]:
split_types = options.BooleanOption(
    True,
    name='split_types',
    help='If True, run the PCA filter separately for each wavelength / '
         'chromophore type instead of jointly.',
)
print('value          :', split_types)
print('used in an if  :', 'yes' if split_types else 'no')   # options are truthy
print('compares to raw:', split_types == True)

split_types.value = 0        # 0/1 are accepted and coerced to False/True
print('after value=0  :', split_types)

try:
    split_types.value = 'yes'
except ValueError as err:
    print('rejected ->', err)

value          : True
used in an if  : yes
compares to raw: True
after value=0  : False
rejected -> split_types: value must be a boolean (got 'yes')


## `StringOption`

Use for text options.  Two flavours:

1. **Free text** - any string is accepted (e.g. a variable name or a label).
2. **Restricted** - pass `allowed=[...]` to limit the value to a known set.
   With `case_sensitive=False` the input is matched case-insensitively and
   normalised to the spelling given in `allowed`.

This is a good fit for the `inputName` / `outputName` fields of the pipeline
modules.

In [9]:
# Free-form string
label = options.StringOption(
    'HbO',
    name='label',
    help='Free-form label attached to the output of this module.',
)
label.value = 'HbR'
print(label)

HbR


In [10]:
# Restricted string, matched case-insensitively
input_name = options.StringOption(
    'od',
    name='inputName',
    allowed=['raw', 'od', 'conc', 'last'],
    case_sensitive=False,
    help="Name of the timeseries this module reads from. 'last' uses the most "
         'recently created timeseries.',
)
input_name.value = 'CONC'          # normalised to the allowed spelling
print('normalised :', input_name)

try:
    input_name.value = 'oxy'
except ValueError as err:
    print('rejected ->', err)

normalised : conc
rejected -> inputName: value must be one of ['raw', 'od', 'conc', 'last'] (got 'oxy')


## `EnumOption`

Use when the option selects one of a fixed set of *named* alternatives that you
want to expose as a proper python type (with autocompletion, `is` comparisons,
and no risk of typos).  Define a `enum.Enum` subclass and hand it to
`EnumOption`.

Values may be assigned as the enum member itself, by **name**, or by **value** -
all three are normalised to the enum member.  Printing shows the member *name*.
`opt.choices` lists everything that is selectable.

In [11]:
class MotionCorrection(enum.Enum):
    """Available motion-correction algorithms."""
    NONE = 0
    SPLINE = 1
    WAVELET = 2
    TDDR = 3


method = options.EnumOption(
    MotionCorrection,
    MotionCorrection.TDDR,
    name='method',
    help='Algorithm used to correct motion artifacts in the optical density '
         'data.',
)

print('value   :', method)                   # prints the member name
print('member  :', method.value)             # the actual enum member
print('choices :', [m.name for m in method.choices])

value   : TDDR
member  : MotionCorrection.TDDR
choices : ['NONE', 'SPLINE', 'WAVELET', 'TDDR']


In [12]:
# All three assignment styles are equivalent
method.value = MotionCorrection.SPLINE       # by member
print(method)
method.value = 'WAVELET'                     # by name
print(method)
method.value = 3                             # by value
print(method)

# The enum member can be used directly in comparisons
if method.value is MotionCorrection.TDDR:
    print('-> running TDDR')

try:
    method.value = 'kalman'
except ValueError as err:
    print('rejected ->', err)

SPLINE
WAVELET
TDDR
-> running TDDR
rejected -> method: value must be a MotionCorrection member ['NONE', 'SPLINE', 'WAVELET', 'TDDR'] (got 'kalman')


## `ChoiceOption`

Use when the allowed values are an explicit, possibly **mixed-type** list and
defining an `Enum` would be overkill - for example an option that accepts a few
keywords *or* `None`.

In [13]:
basis = options.ChoiceOption(
    ['canonical', 'gamma', 'FIR', None],
    value='canonical',
    name='basis',
    help='Hemodynamic response basis used by the GLM. None disables '
         'convolution and fits the raw regressors.',
)
print('value   :', basis)
print('choices :', basis.choices)

basis.value = None                # None is an explicitly allowed choice here
print('value   :', basis)

try:
    basis.value = 'boxcar'
except ValueError as err:
    print('rejected ->', err)

value   : canonical
choices : ['canonical', 'gamma', 'FIR', None]
value   : None
rejected -> basis: value must be one of ['canonical', 'gamma', 'FIR', None] (got 'boxcar')


## `ListOption`

Use for options holding a *list* of values.  You can constrain

* the length via `min_length` / `max_length`, and
* every element via `item_option`, which is simply another option variable
  used as the per-item validator.

Tuples are accepted and converted to lists.

In [14]:
# A list of exactly two positive wavelengths
wavelengths = options.ListOption(
    [760, 850],
    name='wavelengths',
    item_option=options.NumericOption(1, minimum=0, inclusive=False),
    min_length=2,
    max_length=2,
    help='Wavelengths (nm) of the light sources, in the order they appear in '
         'the probe.',
)
print(wavelengths)

wavelengths.value = (690, 830)     # tuples are converted to lists
print(wavelengths)

for bad in ([760, -850], [760]):
    try:
        wavelengths.value = bad
    except ValueError as err:
        print(f'{bad!r} rejected -> {err}')

[760, 850]
[690, 830]
[760, -850] rejected -> wavelengths: item 1: value must be > 0 (got -850)
[760] rejected -> wavelengths: value must have at least 2 entries


In [15]:
# A list of allowed strings, with no length restriction
conditions = options.ListOption(
    ['HbO', 'HbR'],
    name='conditions',
    item_option=options.StringOption('HbO', allowed=['HbO', 'HbR', 'HbT']),
    help='Chromophores included in the group-level model.',
)
print(conditions)
conditions.value = ['HbO', 'HbT']
print(conditions)

['HbO', 'HbR']
['HbO', 'HbT']


## `QuantityOption`

Use for options that carry **physical units**, such as the `fmin` / `fmax`
cut-off frequencies of the `bandpass_filter` module.  Provide the expected
`units`; then

* bare numbers are promoted to that unit (`0.5` -> `0.5 Hz`),
* quantities in compatible units are accepted and converted for the range check
  (e.g. `500 * units.mHz`),
* quantities in incompatible units are rejected (`3 * units.meter`),
* `minimum` / `maximum` are expressed as magnitudes **in `units`**.

In [16]:
fmax = options.QuantityOption(
    1 * units.Hz,
    units=units.Hz,
    minimum=0,
    name='fmax',
    help='Upper cut-off frequency of the band-pass filter.',
)
print('value        :', fmax)

fmax.value = 0.5                       # a bare number is interpreted as Hz
print('from number  :', fmax)

fmax.value = 500 * units.mHz           # compatible units are fine
print('from mHz     :', fmax)

for bad in (-1 * units.Hz, 3 * units.meter):
    try:
        fmax.value = bad
    except ValueError as err:
        print(f'rejected -> {err}')

value        : 1 hertz
from number  : 0.5 hertz
from mHz     : 500 millihertz
rejected -> fmax: value must be >= 0 hertz (got -1 hertz)
rejected -> fmax: value must be convertible to hertz (got <Quantity(3, 'meter')>)


## Help text

Every option accepts a `help=` string (and an optional `description=`).
`print(opt)` deliberately keeps showing just the current value; use
`opt.print_help()` (or `opt.format_help()` to get the string) for the full
documentation of the option - useful for building GUIs and `?`-style help in
the pipeline manager.

In [17]:
fmin = options.QuantityOption(
    0.016 * units.Hz,
    units=units.Hz,
    minimum=0,
    name='fmin',
    description='Lower cut-off frequency',
    help='Frequencies below this value are removed. Set well below the '
         'slowest stimulus frequency to avoid attenuating the response.',
)

fmin.value = 0.01 * units.Hz
print(fmin)          # -> only the value
print()
fmin.print_help()    # -> name, value, description, help and default

0.01 hertz

fmin: 0.01 hertz
  Lower cut-off frequency
  Frequencies below this value are removed. Set well below the slowest stimulus frequency to avoid attenuating the response.
  (default: 0.016 hertz)


In [18]:
# Help for every option defined in this notebook
for opt in (butter_order, ncomp, split_types, input_name, method,
            basis, wavelengths, fmax, fmin):
    opt.print_help()
    print()

butter_order: 4
  Order of the Butterworth filter. Higher values give a sharper roll-off but more ringing.
  (default: 4)

ncomp: 0.8
  Fraction of the variance removed by the PCA filter, or (if >1) the number of components to remove.
  (default: 0.8)

split_types: False
  If True, run the PCA filter separately for each wavelength / chromophore type instead of jointly.
  (default: True)

inputName: conc
  Name of the timeseries this module reads from. 'last' uses the most recently created timeseries.
  (default: od)

method: TDDR
  Algorithm used to correct motion artifacts in the optical density data.
  (default: MotionCorrection.TDDR)

basis: None
  Hemodynamic response basis used by the GLM. None disables convolution and fits the raw regressors.
  (default: canonical)

wavelengths: [690, 830]
  Wavelengths (nm) of the light sources, in the order they appear in the probe.
  (default: [760, 850])

fmax: 500 millihertz
  Upper cut-off frequency of the band-pass filter.
  (default: 1 he

## Custom display

`print(opt)` shows the current value by default.  To customise it you can

1. pass a `formatter=` callable which receives the option and returns a string, or
2. subclass an option type and override `format_value()`.

Both keep the underlying value untouched - only the display changes.

In [19]:
# 1) using a formatter callable
ncomp_pretty = options.NumericOption(
    0.8, name='ncomp', minimum=0, maximum=1,
    help='Fraction of variance removed by the PCA filter.',
    formatter=lambda opt: f'{opt.value:.0%} of the variance',
)
print(ncomp_pretty)
print('raw value:', ncomp_pretty.value)

80% of the variance
raw value: 0.8


In [20]:
# 2) by subclassing
class TimeWindowOption(options.NumericOption):
    """A duration in seconds that displays itself with its unit."""

    def format_value(self):
        return f'{self.value:g} s'


window = TimeWindowOption(
    30, name='window', minimum=0, inclusive=False,
    help='Length of the sliding window used to estimate the baseline.')
print(window)
window.value = 12.5
print(window)

30 s
12.5 s


## Writing your own option type

To support a value that none of the built-in classes covers, subclass
`OptionVariable` and implement `validate()`.  It must **return** the value to
store (so it can also coerce) or raise a `ValueError`/`TypeError`.  Use
`self._error(msg)` to build an error that is automatically prefixed with the
option's name.  Everything else - defaults, `reset()`, help, printing - is
inherited.

In [21]:
class EvenNumberOption(options.OptionVariable):
    """Example custom option: only even integers are allowed."""

    def validate(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise self._error(f'value must be an integer (got {value!r})')
        if value % 2:
            raise self._error(f'value must be even (got {value})')
        return value


nfft = EvenNumberOption(256, name='nfft', help='FFT length; must be even.')
print(nfft)
nfft.value = 512
print(nfft)

try:
    nfft.value = 255
except ValueError as err:
    print('rejected ->', err)

256
512
rejected -> nfft: value must be even (got 255)


## Using them in a module `options` dictionary

The eventual plan is for the pipeline modules to build their `options`
dictionary from these classes.  Because the option variables compare equal to
their raw values and are truthy/falsy like them, they can be introspected by a
GUI while remaining easy to read in the module code (use `.value` when passing
them on to the underlying cedalion function).

*Note: the pipeline modules have **not** been converted yet - this cell only
shows what it will look like.*

In [22]:
example_options = {
    'fmax': options.QuantityOption(1 * units.Hz, units=units.Hz, minimum=0,
                                   name='fmax',
                                   help='Upper cut-off frequency.'),
    'fmin': options.QuantityOption(0.016 * units.Hz, units=units.Hz, minimum=0,
                                   name='fmin',
                                   help='Lower cut-off frequency.'),
    'butter_order': options.NumericOption(4, name='butter_order', minimum=1,
                                          integer_only=True,
                                          help='Butterworth filter order.'),
}

# display like a raw options dict
for key, opt in example_options.items():
    print(f'{key:>14} = {opt}')

print()
# ...and how a module would consume them
print('freq_filter(fmin=%s, fmax=%s, butter_order=%s)' % (
    example_options['fmin'].value,
    example_options['fmax'].value,
    example_options['butter_order'].value,
))

print()
# restore every option to its default
for opt in example_options.values():
    opt.reset()
print('after reset:', {k: str(v) for k, v in example_options.items()})

          fmax = 1 hertz
          fmin = 0.016 hertz
  butter_order = 4

freq_filter(fmin=0.016 hertz, fmax=1 hertz, butter_order=4)

after reset: {'fmax': '1 hertz', 'fmin': '0.016 hertz', 'butter_order': '4'}
